In [6]:
import pandas as pd
sap_final = pd.read_json(r"C:\Users\mehat\OneDrive - SRH\Subject\NLP\data\sap_news.json")
gnews_df = pd.read_json(r"C:\Users\mehat\OneDrive - SRH\Subject\NLP\data\gnews_articles.json")

In [7]:
irrelevant_titles = [
    "What is dragon’s blood resin? The forgotten 2,000-year-old skincare ingredient used by ancient Roman and Arab women",
    "Tech updates (June 11, 2026): New ASUS gaming laptops, Canva Offline, Samsung AI deals",
    "Business wisdom of the day: 'The bamboo that bends is stronger than the oak that resists'",
    "India emerges as world's top marketing"
]

gnews_final = gnews_df[
    ~gnews_df["title"].isin(irrelevant_titles)
]

In [8]:
gnews_final = (
    gnews_final.assign(title_key=gnews_final["title"].str[:20])
            .drop_duplicates(subset=["title_key"])
            .drop(columns=["title_key"])
)

In [9]:
all_docs = pd.concat(
    [sap_final, gnews_final],
    ignore_index=True
)

print(all_docs.shape)

(56, 9)


In [10]:
all_docs["content"] = all_docs["content"].fillna(
    all_docs["final_content"]
)

In [11]:
cols_to_drop = [
    "link",
    "full_content",
    "final_content"
]

all_docs = all_docs.drop(
    columns=cols_to_drop,
    errors="ignore"
)

In [12]:
all_docs["text"] = (
    all_docs["title"].fillna("")
    + " "
    + all_docs["description"].fillna("")
    + " "
    + all_docs["content"].fillna("")
)

In [13]:
all_docs.shape

(56, 7)

In [14]:
import re

def clean_text(text):

    text = str(text)

    # Unicode cleanup
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", "")

    # Remove GNews truncation markers
    text = re.sub(r"\[\+\d+\schars\]", "", text)

    # Remove repeated separators
    text = re.sub(r"\*{3,}", " ", text)
    text = re.sub(r">{3,}", " ", text)
    text = re.sub(r"-{3,}", " ", text)
    text = re.sub(r"={3,}", " ", text)

    # Normalize whitespace/newlines
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [15]:
all_docs["text"] = all_docs["text"].apply(clean_text)

In [16]:
import re

def find_non_ascii(text):
    return re.findall(r'[^\x00-\x7F]', str(text))

non_ascii = []

for text in all_docs["text"]:
    non_ascii.extend(find_non_ascii(text))

from collections import Counter

Counter(non_ascii).most_common(50)

[('’', 288),
 ('”', 216),
 ('“', 213),
 ('—', 101),
 ('‑', 33),
 ('–', 31),
 ('‘', 18),
 ('®', 12),
 ('\u2060', 12),
 ('™', 6),
 ('©', 5),
 ('€', 5),
 ('\u200c', 5),
 ('…', 4),
 ('ö', 3),
 ('é', 3),
 ('ı', 3),
 ('á', 2),
 ('×', 1),
 ('ú', 1),
 ('ü', 1),
 ('Ö', 1),
 ('ş', 1)]

In [18]:
import re

patterns = []

for text in all_docs["text"]:
    patterns.extend(
        re.findall(r'([*#=><\-]{2,})', str(text))
    )

from collections import Counter

Counter(patterns).most_common(20)

[('--', 13), ('**', 2)]

In [19]:
import re

urls = []

for text in all_docs["text"]:
    urls.extend(
        re.findall(r'https?://\S+', str(text))
    )

print("Total URLs:", len(urls))
print(urls[:20])

Total URLs: 8
['https://www.sap.com/copyright', 'https://www.sap.com/copyright', 'https://www.sap.com/copyright', 'https://www.sap.com/copyright', 'https://www.sap.com/copyright', 'https://www.junkiescoder.com/', 'https://mma.prnewswire.com/media/2996900/Junkies_Coder.jpg', 'https://www.merixstudio.com/services/enterprise-software-development.']


In [20]:
emails = []

for text in all_docs["text"]:
    emails.extend(
        re.findall(r'[\w\.-]+@[\w\.-]+', str(text))
    )

set(emails)

{'aimee.leabon@sap.com',
 'belen.martinez@sap.com',
 'contactus@intelegain.com',
 'dana.roesiger@sap.com',
 'daniel.reinhardt@sap.com',
 'ekin.tayali@sap.com',
 'pr.error.rectification@gmail.com.',
 'press@sap.com',
 'ulrika.wass@sap.com',
 'universityalliances@sap.com'}

In [21]:
truncations = []

for text in all_docs["text"]:
    truncations.extend(
        re.findall(r'\[\+\d+\schars\]', str(text))
    )

Counter(truncations)

Counter()

In [22]:
for idx, row in all_docs.iterrows():

    text = row["text"]

    if any(
        pattern in text
        for pattern in [
            "\xa0",
            "\u200b",
            "*****",
            ">>>>",
            "[+"
        ]
    ):
        print("="*80)
        print(idx)
        print(row["title"])
        print(text[:1000])

In [ ]:
def clean_text(text):

    text = str(text)

    # remove invisible unicode chars
    text = text.replace("\u2060", "")
    text = text.replace("\u200c", "")
    text = text.replace("\u200b", "")
    text = text.replace("\xa0", " ")

    # remove urls
    text = re.sub(r'https?://\S+', '', text)

    # remove emails
    text = re.sub(
        r'[\w\.-]+@[\w\.-]+\.\w+',
        '',
        text
    )

    # remove GNews truncation
    text = re.sub(
        r'\[\+\d+\schars\]',
        '',
        text
    )

    # normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [24]:
all_docs["clean_text"] = (
    all_docs["text"]
    .apply(clean_text)
)

In [25]:
all_docs["clean_length"] = (
    all_docs["clean_text"]
    .str.len()
)

all_docs["clean_length"].describe()

count       56.000000
mean      5034.964286
std       2762.852630
min        651.000000
25%       3233.000000
50%       5077.500000
75%       6168.750000
max      17551.000000
Name: clean_length, dtype: float64

In [26]:
all_docs.to_json(
    r"C:\Users\mehat\OneDrive - SRH\Subject\NLP\data\clean_data.json",
    orient="records",
    indent=4
)